# Fock Multi-ξ PARF at Full V_φ Capacity (H=128)

## Motivation

Multi-ξ PARF at H=128 (gathered V_φ + Level-2 checkpointing) reached
**12.06 PPL** with log-spaced α (K=8) — closing 77% of the gap between the
multi-ξ SPLM baseline (14.69 PPL) and the attention baseline (7.81 PPL).

This notebook asks: **can Fock-space latent registers close the remaining
gap?**  The Fock mechanism adds M register particles with creation/destruction
gates, enabling the model to learn long-range dependencies that escape the
v0 expressivity ceiling (Dyck_n recognition).

Two gate variants are swept:

| Variant | Creation gate | Reverse channel | Key idea |
|---------|-------------|-----------------|----------|
| **v1** | Mean-conditioned MLP | — | Simple; one gate per layer |
| **v2** | Q/K/V cross-attention | Optional non-conservative Q_i | Richer content routing, register-to-token force |

## Baselines

| Model | PPL | Steps | Notes |
|-------|-----|-------|-------|
| Attention baseline | 7.81 | 8000 | Target |
| Multi-ξ PARF H=128 (best) | 12.06 | 8000 | Log-spaced α, K=8 |
| Multi-ξ SPLM (no PARF) | 14.69 | 4000 | V_θ only |

## Arms (13-arm full sweep)

| # | Arm | Fock | K | M | Discipline | Rev | k | Steps | Description |
|---|-----|------|---|---|-----------|-----|---|-------|-------------|
| 1 | `v1_K4_M16_lifo` | v1 | 4 | 16 | LIFO | — | 8 | 8k | Baseline v1 |
| 2 | `v1_K4_M32_lifo` | v1 | 4 | 32 | LIFO | — | 8 | 8k | Register capacity test |
| 3 | `v1_K4_M16_free` | v1 | 4 | 16 | free | — | 8 | 8k | Discipline ablation |
| 4 | `v2_K4_M16_lifo` | v2 | 4 | 16 | LIFO | ✓ | 8 | 8k | Baseline v2 |
| 5 | `v2_K4_M16_no_rev` | v2 | 4 | 16 | LIFO | ✗ | 8 | 8k | Reverse channel ablation |
| 6 | `v2_K4_M32_lifo` | v2 | 4 | 32 | LIFO | ✓ | 8 | 8k | Register capacity (v2) |
| 7 | `v2_K2_M16_lifo` | v2 | 2 | 16 | LIFO | ✓ | 8 | 8k | Cheaper channels |
| 8 | `v2_K4_M16_k4` | v2 | 4 | 16 | LIFO | ✓ | 4 | 8k | Routing density |
| 9 | `v1_K4_M16_struct` | v1 | 4 | 16 | LIFO | — | 8 | 8k | Non-competitive V_φ |
| 10 | `v2_K4_M8_lifo` | v2 | 4 | 8 | LIFO | ✓ | 8 | 8k | Minimal registers |
| 11 | `v2_K4_M4_lifo` | v2 | 4 | 4 | LIFO | ✓ | 8 | 8k | Tiny pool (interference test) |
| 12 | `v2_K4_M16_lifo_16k` | v2 | 4 | 16 | LIFO | ✓ | 8 | **16k** | Convergence test |
| 13 | `v2_K4_M16_lifo_32k` | v2 | 4 | 16 | LIFO | ✓ | 8 | **32k** | Extended convergence test |

## Hardware

- **A100 40GB / H100 80GB**: Fock adds ~2–32K params overhead per arm.
  Register concatenation extends T→T+M per layer, but gathered V_φ keeps
  pair-interaction cost at O((T+M)·k).  Expect ~1.2–1.5× wall-time vs
  non-Fock multi-ξ PARF.
- No grad-accum needed
- TF32 disabled for autograd.grad stability

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/paper_nmi_fock_multixi_h128')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'paper_nmi_fock_multixi_h128'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dimitarpg13/paper_nmi.git'
REPO_DIR = REPO_PARENT / 'paper_nmi'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets',
                    'transformers', 'tokenizers', 'huggingface_hub', 'pyarrow'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
PARF_DIR    = REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for PARF autograd.grad stability')
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

All arms use the memory-optimized Fock Multi-ξ PARF:
- `--use-layer-checkpoint` (Level-2: O(L) → O(1) layer memory)
- `--use-gathered-v-phi`  (Stage-1.5b: O(T²) → O(T·k) V_φ memory)
- `--v-phi-phi-hidden 128 --v-phi-theta-hidden 128` (full V_φ capacity)
- `--ln-before-distance --per-layer-v-phi-scale` (P8 patches)
- `--xi-alpha-init-mode log_spaced` (auto-tuned α, the H=128 winner)
- `--gumbel-tau-min 0.3` (raised floor to prevent STE gradient explosions)
- `--fock-grad-clip 0.5` (separate tighter clip for Fock-specific params)
- No `--grad-accum` needed

In [ ]:
MODE          = 'scaleup'     # 8000 steps for fully converged results
FIXED_GAMMA   = 0.30
SEED          = 0
MAX_TRAIN_TOK = 5_000_000
V_PHI_H       = 128           # full V_phi capacity
GUMBEL_TAU_MIN = 0.3          # raised from 0.1 to prevent STE gradient explosions
FOCK_GRAD_CLIP = 0.5          # separate tighter grad clip for Fock params

ARM_DEFS = {
    # ── v1 arms ──
    'v1_K4_M16_lifo': {
        'fock_version': 'v1',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': False,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v1, K=4 log-spaced, M=16, LIFO, k=8 — baseline v1 with multi-xi',
    },
    'v1_K4_M32_lifo': {
        'fock_version': 'v1',
        'n_registers': 32,
        'stack_discipline': True,
        'reverse_channel': False,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v1, K=4 log-spaced, M=32, LIFO, k=8 — register capacity test',
    },
    'v1_K4_M16_free': {
        'fock_version': 'v1',
        'n_registers': 16,
        'stack_discipline': False,
        'reverse_channel': False,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v1, K=4 log-spaced, M=16, free (no LIFO), k=8 — discipline ablation',
    },
    # ── v2 arms ──
    'v2_K4_M16_lifo': {
        'fock_version': 'v2',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v2 (Q/K/V + reverse), K=4 log-spaced, M=16, LIFO, k=8 — baseline v2',
    },
    'v2_K4_M16_no_rev': {
        'fock_version': 'v2',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': False,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v2 (Q/K/V, no reverse), K=4 log-spaced, M=16, LIFO, k=8 — reverse ablation',
    },
    'v2_K4_M32_lifo': {
        'fock_version': 'v2',
        'n_registers': 32,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v2, K=4 log-spaced, M=32, LIFO, k=8 — register capacity (v2)',
    },
    'v2_K2_M16_lifo': {
        'fock_version': 'v2',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 2,
        'xi_alpha_init_mode': 'explicit',
        'xi_alpha_inits': '0.50,0.95',
        'desc': 'v2, K=2 explicit, M=16, LIFO, k=8 — cheaper channel test',
    },
    'v2_K4_M16_k4': {
        'fock_version': 'v2',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 4,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v2, K=4 log-spaced, M=16, LIFO, k=4 — routing density',
    },
    'v1_K4_M16_struct': {
        'fock_version': 'v1',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': False,
        'v_phi_kind': 'structural',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v1, K=4 log-spaced, M=16, LIFO, k=8, plain structural V_\u03c6 — competitive gate ablation',
    },
    'v2_K4_M8_lifo': {
        'fock_version': 'v2',
        'n_registers': 8,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v2, K=4 log-spaced, M=8, LIFO, k=8 — minimal registers',
    },
    'v2_K4_M4_lifo': {
        'fock_version': 'v2',
        'n_registers': 4,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'v2, K=4 log-spaced, M=4, LIFO, k=8 — tiny register pool (interference test)',
    },
    'v2_K4_M16_lifo_16k': {
        'fock_version': 'v2',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 16000,
        'desc': 'v2, K=4 log-spaced, M=16, LIFO, k=8 — 16k steps (convergence test)',
    },
    'v2_K4_M16_lifo_32k': {
        'fock_version': 'v2',
        'n_registers': 16,
        'stack_discipline': True,
        'reverse_channel': True,
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 32000,
        'desc': 'v2, K=4 log-spaced, M=16, LIFO, k=8 — 32k steps (extended convergence test)',
    },
}

ALL_ARMS = list(ARM_DEFS.keys())

print(f'Arms defined : {len(ALL_ARMS)}')
for name in ALL_ARMS:
    print(f'  {name}')

In [ ]:
# ── SELECT ARMS TO RUN ───────────────────────────────────────────────────────
# Edit ARMS_TO_RUN to target any subset of the arms defined above.
# Run this cell after Cell 7 every time you change the selection.
#
# Examples:
#   ARMS_TO_RUN = ALL_ARMS                               # full sweep (default)
#   ARMS_TO_RUN = ['v1_K4_M16_lifo']                    # single arm (re-run / debug)
#   ARMS_TO_RUN = ['v2_K4_M16_lifo', 'v2_K4_M32_lifo'] # two arms
#   ARMS_TO_RUN = [n for n in ALL_ARMS if n.startswith('v2')]  # all v2 arms
# ─────────────────────────────────────────────────────────────────────────────

# Initialise results dict if this is a fresh run
if "results" not in dir():
    results = {}

ARMS_TO_RUN = ALL_ARMS          # ← change this to run a subset

# Validate
invalid = [n for n in ARMS_TO_RUN if n not in ARM_DEFS]
if invalid:
    raise ValueError(f'Unknown arm names: {invalid}\nValid: {ALL_ARMS}')

# Summary
print(f'Memory mode  : Level-2 checkpoint + Stage-1.5b gathered V_\u03c6')
print(f'V_\u03c6 capacity : H={V_PHI_H}')
print(f'Stability    : gumbel_tau_min={GUMBEL_TAU_MIN}  fock_grad_clip={FOCK_GRAD_CLIP}')
_n_steps = {"scaleup": 8000, "pilot": 4000, "smoke": 300}.get(MODE, "???")
print(f'Schedule     : {MODE}  ({_n_steps} steps)')
print()
print(f'Selected {len(ARMS_TO_RUN)} / {len(ALL_ARMS)} arms:')
for name in ARMS_TO_RUN:
    d = ARM_DEFS[name]
    disc = 'LIFO' if d['stack_discipline'] else 'free'
    rev = '+rev' if d.get('reverse_channel') else ''
    steps_str = f'  steps={d["max_steps"]}' if d.get('max_steps') else ''
    print(f'  {name:25s}  Fock-{d["fock_version"]}{rev}  K={d["xi_channels"]}  '
          f'M={d["n_registers"]}  {disc}  k={d["top_k"]}  V_\u03c6={d["v_phi_kind"]}{steps_str}')
    print(f'  {"":25s}  {d["desc"]}')



## 4. Precompute logfreq surprisal (if needed)

In [ ]:
LOGFREQ_PATH = SCRIPTS_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'

if not LOGFREQ_PATH.exists():
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / 'compute_unigram_frequencies_tinystories.py')],
        cwd=str(SCRIPTS_DIR), check=True,
    )
    assert LOGFREQ_PATH.exists()
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')

## 5. Train Fock Multi-ξ PARF arms (H=128, gathered V_φ)

Each arm trains a `FockMultiXiPARFLM` with:
- Full V_φ capacity (H=128)
- Level-2 per-layer checkpointing
- Stage-1.5b gathered V_φ evaluation
- No gradient accumulation
- Fock register pool (v1 or v2 gates)

Completed arms are skipped on re-run.

In [ ]:
import re
from IPython.display import clear_output

# ── Log-line parsers (match [fock-multixi-parf] prefixed stdout lines) ────────
_TRAIN_RE = re.compile(
    r'step\s+(\d+)/(\d+)\s+train\s+([\d.]+)\s+lr\s+([\d.eE+-]+)\s+'
    r'grad\s+([\d.]+)\s+'
    r'(?:\(fock=([\d.]+)\s+vphi=([\d.]+)\)\s+)?'
    r'gamma=([\d.]+)\s+tau=([\d.]+)'
    r'(?:\s+fock_tau=([\d.]+))?'
    r'(?:\s+rev_s=([\d.eE+-]+))?'
    r'\s+[^\[]*\[([^\]]+)\]\s+elapsed\s+([\d.]+)s'
)
_EVAL_RE = re.compile(
    r'>>>\s+eval\s+@\s+(\d+):\s+val\s+([\d.]+)\s+ppl\s+([\d.]+)'
)

def _parse_train(line):
    m = _TRAIN_RE.search(line)
    if not m:
        return None
    d = {
        'step': int(m.group(1)), 'total': int(m.group(2)),
        'train_loss': float(m.group(3)), 'lr': float(m.group(4)),
        'grad': float(m.group(5)),
        'gamma': float(m.group(8)), 'tau': float(m.group(9)),
        'alphas': m.group(12),
        'elapsed_s': float(m.group(13)),
    }
    if m.group(6) is not None:
        d['grad_fock'] = float(m.group(6))
    if m.group(7) is not None:
        d['grad_vphi'] = float(m.group(7))
    if m.group(10) is not None:
        d['fock_tau'] = float(m.group(10))
    if m.group(11) is not None:
        d['rev_s'] = float(m.group(11))
    return d

def _parse_eval(line):
    m = _EVAL_RE.search(line)
    if not m:
        return None
    return {'step': int(m.group(1)), 'val_loss': float(m.group(2)),
            'val_ppl': float(m.group(3))}

def _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines, done=False):
    clear_output(wait=True)
    import sys as _sys; _sys.stdout.flush()
    status = '\u2713 DONE' if done else '\u2699 RUNNING'
    disc = 'LIFO' if arm_def['stack_discipline'] else 'free'
    rev = '+rev' if arm_def.get('reverse_channel') else ''
    W = 65
    print('\u2550' * W)
    print(f'  {status}  {arm_name}')
    print(f'  Fock-{arm_def["fock_version"]}{rev}  K={arm_def["xi_channels"]}  '
          f'M={arm_def["n_registers"]}  {disc}  k={arm_def["top_k"]}')
    print(f'  {arm_def["desc"]}')
    print('\u2550' * W)

    if tr:
        step, total = tr['step'], tr['total']
        elapsed = time.time() - t_start
        train_elapsed = (time.time() - t_train_start) if t_train_start else elapsed
        pct = 100 * step / total
        eta_s = train_elapsed * (total - step) / step if step > 0 else 0
        eta_str = 'Done!' if done else f'ETA {eta_s / 60:.1f} min'
        bar_w = 52
        filled = int(bar_w * pct / 100)
        bar = '\u2588' * filled + '\u2591' * (bar_w - filled)
        print(f'  [{bar}]')
        print(f'  Step : {step:5d} / {total}  ({pct:.1f}%)'
              f'   Elapsed: {elapsed / 60:.1f} min   {eta_str}')
        print()
        grad_detail = ''
        if 'grad_fock' in tr:
            grad_detail = f'  (fock={tr["grad_fock"]:.1f}  vphi={tr["grad_vphi"]:.1f})'
        print(f'  train loss : {tr["train_loss"]:.4f}'
              f'   lr : {tr["lr"]:.2e}'
              f'   grad norm : {tr["grad"]:.3f}{grad_detail}')
        fock_tau_str = ''
        if 'fock_tau' in tr:
            fock_tau_str = f'   fock \u03c4_create: {tr["fock_tau"]:.4f}'
        rev_str = ''
        if 'rev_s' in tr:
            rev_str = f'   rev scale: {tr["rev_s"]:.3f}'
        print(f'  \u03b3          : {tr["gamma"]:.4f}'
              f'   \u03c4 (Gumbel): {tr["tau"]:.4f}'
              f'{fock_tau_str}{rev_str}')
        print(f'  \u03b1 channels : [{tr["alphas"]}]')

    if ev:
        print()
        print(f'  Last val PPL : {ev["val_ppl"]:.3f}'
              f'   val loss : {ev["val_loss"]:.4f}'
              f'   (@ step {ev["step"]})')

    noise = [l for l in recent_lines
             if l.strip()
             and '[fock-multixi-parf] step' not in l
             and not l.startswith('{')]
    if noise:
        print()
        print('  Recent output:')
        for l in noise[-6:]:
            print(f'    {l}')

def _run_arm_streaming(arm_name, arm_def, cmd):
    """Launch Fock trainer, stream stdout, show live progress. Returns (returncode, all_lines)."""
    all_lines, recent_lines = [], []
    tr, ev = None, None
    t_start = time.time()
    t_train_start = None  # set on first training step line

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        cwd=str(SCRIPTS_DIR),
        env={**__import__("os").environ, "PYTHONUNBUFFERED": "1"},
    )
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            all_lines.append(line)
            recent_lines = all_lines[-40:]

            new_tr = _parse_train(line)
            new_ev = _parse_eval(line)
            if new_tr:
                tr = new_tr
                if t_train_start is None:
                    t_train_start = time.time()  # first step: ignore tokenisation time
            if new_ev:
                ev = new_ev
            if new_tr or new_ev:
                _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines)
            elif tr is None:
                if line.strip() and not line.startswith('{'):
                    print(line, flush=True)
    except Exception as exc:
        print(f'\n  STREAM ERROR: {exc}')
    finally:
        proc.wait()

    _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines, done=True)
    return proc.returncode, all_lines


# ── Training loop ─────────────────────────────────────────────────────────────
TRAINER = str(SCRIPTS_DIR / 'train_fock_multixi_scaleup.py')

results = {}

for arm_name in ARMS_TO_RUN:
    arm_def = ARM_DEFS[arm_name]
    arm_results_dir = DRIVE_RESULTS / arm_name
    arm_results_dir.mkdir(parents=True, exist_ok=True)

    # Skip if already complete
    summary_glob = list(arm_results_dir.glob('*_summary.md'))
    if summary_glob:
        print(f'\n⏭  {arm_name}: SKIP (already complete)')
        with open(summary_glob[0]) as f:
            for line in f:
                if 'Final' in line:
                    print(f'   {line.strip()}')
        results[arm_name] = {'status': 'skipped'}
        continue

    cmd = [
        sys.executable, "-u", TRAINER,
        '--mode', MODE,
        '--seed', str(SEED),
        '--fixed-gamma', str(FIXED_GAMMA),
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--results-dir', str(arm_results_dir),
        '--tag-suffix', arm_name,
        '--logfreq-path', str(LOGFREQ_PATH),
        '--device', DEVICE,
        '--v-phi-kind', arm_def['v_phi_kind'],
        '--top-k', str(arm_def['top_k']),
        '--v-phi-phi-hidden', str(V_PHI_H),
        '--v-phi-theta-hidden', str(V_PHI_H),
        '--use-layer-checkpoint',
        '--use-gathered-v-phi',
        '--ln-before-distance',
        '--per-layer-v-phi-scale',
        '--gumbel-tau-min', str(GUMBEL_TAU_MIN),
        '--fock-grad-clip', str(FOCK_GRAD_CLIP),
        '--xi-channels', str(arm_def['xi_channels']),
        '--xi-alpha-init-mode', arm_def['xi_alpha_init_mode'],
        '--fock-version', arm_def['fock_version'],
        '--n-registers', str(arm_def['n_registers']),
    ]
    if arm_def.get('xi_alpha_inits') is not None:
        cmd.extend(['--xi-alpha-inits', arm_def['xi_alpha_inits']])
    if arm_def['stack_discipline']:
        cmd.append('--stack-discipline')
    else:
        cmd.append('--no-stack-discipline')
    if arm_def['fock_version'] == 'v2':
        cmd.append('--reverse-channel' if arm_def.get('reverse_channel', True)
                   else '--no-reverse-channel')
    if arm_def.get('max_steps') is not None:
        cmd.extend(['--max-steps', str(arm_def['max_steps'])])

    rc, all_lines = _run_arm_streaming(arm_name, arm_def, cmd)

    if rc != 0:
        print(f'\n  ✗ FAILED: trainer exited with code {rc}')
        print('  Last 30 lines of output:')
        for l in all_lines[-30:]:
            print(f'    {l}')
        results[arm_name] = {'status': 'failed', 'returncode': rc}
        continue

    summary_files = list(arm_results_dir.glob('*_summary.md'))
    if summary_files:
        print('\n  ── Training summary ──')
        with open(summary_files[0]) as f:
            print(f.read())

    results[arm_name] = {'status': 'completed'}

# Final status table
print(f'\n{"═" * 65}')
print(f'All selected arms finished  ({len(ARMS_TO_RUN)} selected)')
for name, r in results.items():
    sym = {'completed': '✓', 'skipped': '⏭', 'failed': '✗'}.get(r['status'], '?')
    print(f'  {sym}  {name:30s}  {r["status"]}')


## 6. Results comparison

Compare Fock Multi-ξ PARF arms against:
- Multi-ξ PARF H=128 (best): 12.06 PPL
- Multi-ξ SPLM (no PARF): 14.69 PPL
- Attention baseline: 7.81 PPL

In [ ]:
import matplotlib.pyplot as plt

BASELINES = {
    'Multi-\u03be PARF H=128 (best)': 12.06,
    'Multi-\u03be SPLM (no PARF)': 14.69,
    'Multi-\u03be PARF H=16 (old)': 15.44,
    'Attention baseline': 7.81,
}

arm_ppls = {}
arm_alphas = {}
arm_details = {}

for arm_name in ARM_DEFS:
    arm_dir = DRIVE_RESULTS / arm_name
    ckpt_files = list(arm_dir.glob('*_ckpt_latest.pt'))
    if not ckpt_files:
        continue
    ckpt = torch.load(ckpt_files[0], map_location='cpu', weights_only=False)
    arm_ppls[arm_name] = ckpt.get('final_val_ppl')
    arm_alphas[arm_name] = ckpt.get('final_xi_alphas')
    arm_details[arm_name] = {
        'n_params': ckpt.get('n_params'),
        'n_v_phi': ckpt.get('n_v_phi_params'),
        'n_fock': ckpt.get('n_fock_overhead_params'),
        'fock_version': ckpt.get('fock_version'),
        'n_registers': ckpt.get('n_registers'),
        'elapsed_sec': ckpt.get('elapsed_sec'),
    }

if not arm_ppls:
    print('No results found yet.')
else:
    print(f'{"Arm":30s} {"Fock":>5s} {"K":>3s} {"M":>3s} {"k":>3s} '
          f'{"Fock OH":>8s} {"Total":>10s} '
          f'{"Final \u03b1":35s} {"PPL":>8s}')
    print('\u2500' * 115)
    for name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        alpha_str = ', '.join(f'{a:.4f}' for a in arm_alphas[name]) if arm_alphas[name] else '?'
        d = ARM_DEFS[name]
        det = arm_details[name]
        fock_oh = det.get('n_fock') or 0
        total = det.get('n_params') or 0
        print(f'{name:30s} {d["fock_version"]:>5s} {d["xi_channels"]:3d} '
              f'{d["n_registers"]:3d} {d["top_k"]:3d} '
              f'{fock_oh:8,d} {total:10,d} '
              f'[{alpha_str:33s}] {arm_ppls[name]:8.2f}')
    print('\u2500' * 115)
    for bname, bppl in BASELINES.items():
        print(f'{bname:30s} {"":>5s} {"":>3s} {"":>3s} {"":>3s} '
              f'{"":>8s} {"":>10s} '
              f'{"":35s} {bppl:8.2f}  (baseline)')

    best_arm = min(arm_ppls, key=lambda x: arm_ppls[x])
    best_ppl = arm_ppls[best_arm]
    best_def = ARM_DEFS[best_arm]
    print(f'\nBest arm: {best_arm} (Fock-{best_def["fock_version"]}, '
          f'M={best_def["n_registers"]}) \u2192 {best_ppl:.2f} PPL')
    if best_ppl < 12.06:
        print(f'  \u2713 Beats multi-\u03be PARF H=128 (12.06) by {12.06 - best_ppl:.2f} PPL')
        print(f'  \u2192 Fock registers ADD VALUE on top of multi-\u03be PARF!')
    else:
        print(f'  Still above multi-\u03be PARF H=128 (12.06) by '
              f'{best_ppl - 12.06:.2f} PPL')
    gap_closed = (14.69 - best_ppl) / (14.69 - 7.81) * 100
    print(f'  Gap closed (SPLM\u2192Attn): {gap_closed:.1f}%')

In [ ]:
if arm_ppls:
    all_names = (list(sorted(arm_ppls, key=lambda x: arm_ppls[x]))
                 + list(BASELINES.keys()))
    all_ppls  = ([arm_ppls[n] for n in sorted(arm_ppls, key=lambda x: arm_ppls[x])]
                 + list(BASELINES.values()))

    n_arms = len(arm_ppls)
    arm_colors = []
    for n in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        if ARM_DEFS[n]['fock_version'] == 'v1':
            arm_colors.append('#2ecc71')
        else:
            arm_colors.append('#9b59b6')
    colors = arm_colors + ['#e67e22', '#3498db', '#e74c3c', '#95a5a6']

    fig, ax = plt.subplots(figsize=(14, max(6, len(all_names) * 0.6)))
    bars = ax.barh(all_names, all_ppls, color=colors, edgecolor='white')
    for bar, ppl in zip(bars, all_ppls):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{ppl:.2f}', va='center', fontsize=9)

    ax.axvline(x=12.06, color='#e67e22', linestyle='--', linewidth=1.5,
               label='Multi-\u03be PARF H=128 best (12.06)')
    ax.axvline(x=14.69, color='#3498db', linestyle=':', linewidth=1.5,
               label='Multi-\u03be SPLM baseline (14.69)')
    ax.set_xlabel('Val PPL (lower is better)')
    ax.set_title(f'Fock Multi-\u03be PARF H=128 \u2014 '
                 f'{MODE} mode (v1=green, v2=purple)')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'fock_multixi_parf_h128_comparison.png', dpi=150)
    plt.show()

    # Loss curves overlay
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    for arm_name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        arm_dir = DRIVE_RESULTS / arm_name
        log_files = list(arm_dir.glob('*_training_log.jsonl'))
        if not log_files:
            continue
        steps, vals = [], []
        with open(log_files[0]) as f:
            for line in f:
                rec = json.loads(line)
                if 'val_ppl' in rec:
                    steps.append(rec['step'])
                    vals.append(rec['val_ppl'])
        if steps:
            ax2.plot(steps, vals, marker='o', markersize=4,
                     label=f'{arm_name} ({arm_ppls[arm_name]:.2f})')

    ax2.axhline(y=12.06, color='#e67e22', linestyle='--', linewidth=1.5,
                label='Multi-\u03be PARF H=128 (12.06)')
    ax2.axhline(y=14.69, color='#3498db', linestyle=':', linewidth=1.5,
                label='Multi-\u03be SPLM (14.69)')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Val PPL')
    ax2.set_title('Convergence: Fock Multi-\u03be PARF H=128 arms')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc='upper right')
    fig2.tight_layout()
    fig2.savefig(DRIVE_RESULTS / 'fock_multixi_parf_h128_convergence.png', dpi=150)
    plt.show()

## 7. Save consolidated report

In [ ]:
report = {
    'experiment': 'fock_parf_multixi_h128_gathered',
    'memory_mode': 'Level-2 checkpoint + Stage-1.5b gathered V_phi',
    'v_phi_hidden': V_PHI_H,
    'config': {
        'mode': MODE,
        'fixed_gamma': FIXED_GAMMA,
        'seed': SEED,
        'grad_accum': 1,
        'use_layer_checkpoint': True,
        'use_gathered_v_phi': True,
    },
    'arms': {},
    'baselines': BASELINES,
}

for name in arm_ppls:
    d = ARM_DEFS[name]
    report['arms'][name] = {
        'fock_version': d['fock_version'],
        'n_registers': d['n_registers'],
        'stack_discipline': d['stack_discipline'],
        'reverse_channel': d.get('reverse_channel'),
        'v_phi_kind': d['v_phi_kind'],
        'top_k': d['top_k'],
        'xi_channels': d['xi_channels'],
        'xi_alpha_init_mode': d['xi_alpha_init_mode'],
        'final_ppl': arm_ppls[name],
        'final_alphas': arm_alphas.get(name),
        'n_params': arm_details.get(name, {}).get('n_params'),
        'n_v_phi_params': arm_details.get(name, {}).get('n_v_phi'),
        'n_fock_params': arm_details.get(name, {}).get('n_fock'),
        'elapsed_sec': arm_details.get(name, {}).get('elapsed_sec'),
    }

report_path = DRIVE_RESULTS / 'fock_parf_multixi_h128_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved: {report_path}')
if IN_COLAB:
    print('Results are persisted on Google Drive at:')
    print(f'  {DRIVE_ROOT}')